# Exploratory Data Analysis (EDA)
## Harga Komoditas Pangan Aceh (2021-2026)

Notebook ini menampilkan hasil analisis eksplorasi data harga komoditas pangan di Provinsi Aceh.

**Dataset:** File JSON tahunan dari folder `../dataup/data/`

---

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES (jalankan sekali saja)
# ============================================================
!pip install pandas numpy matplotlib seaborn openpyxl scipy

In [ ]:
# ============================================================
# SETUP & IMPORT LIBRARIES
# ============================================================
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown, HTML

# Konfigurasi
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.figsize': (14, 6), 'figure.dpi': 120})

# Path ke folder data
DATA_DIR = Path('..') / 'dataup' / 'data'
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Files: {[f.name for f in sorted(DATA_DIR.glob("*.json"))]}')
print('\nSetup selesai!')

---
## 1. Data Loading
Membaca seluruh file JSON dan menggabungkan menjadi satu DataFrame.

In [ ]:
def parse_harga(val):
    """Konversi string harga Indonesia '15,650' -> float 15650.0"""
    if val is None or val == '' or val == '-':
        return np.nan
    try:
        return float(str(val).replace(',', ''))
    except (ValueError, TypeError):
        return np.nan

# Load semua file JSON
json_files = sorted(DATA_DIR.glob('*.json'))
frames = []
for fp in json_files:
    with open(fp, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    if isinstance(raw, list):
        df_temp = pd.json_normalize(raw)
    elif isinstance(raw, dict):
        for v in raw.values():
            if isinstance(v, list):
                df_temp = pd.json_normalize(v)
                break
        else:
            df_temp = pd.json_normalize([raw])
    else:
        df_temp = pd.DataFrame([raw])
    df_temp['_source_file'] = fp.name
    frames.append(df_temp)
    print(f'  [OK] {fp.name}: {len(df_temp):,} baris')

df_raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal data gabungan: {len(df_raw):,} baris, {len(df_raw.columns)} kolom')

---
## 2. Data Cleaning & Transformasi

In [ ]:
df = df_raw.copy()

# Konversi harga string -> numeric
df['harga_numeric'] = df['harga'].apply(parse_harga)

# Konversi tanggal
df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')
df['tahun'] = df['tanggal'].dt.year
df['bulan'] = df['tanggal'].dt.month
df['bulan_tahun'] = df['tanggal'].dt.to_period('M')

# Strip whitespace pada kolom string
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

print('Data cleaning selesai.')
print(f'Shape: {df.shape}')
print(f'Kolom: {list(df.columns)}')
df.head(10)

---
## 3. Informasi Dataset

In [ ]:
print('='*60)
print('INFORMASI DATASET')
print('='*60)
print(f'  Jumlah File JSON    : {len(json_files)}')
print(f'  Total Baris         : {len(df):,}')
print(f'  Total Kolom         : {len(df.columns)}')
print(f'  Memory Usage        : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'  Duplicate Rows      : {df.duplicated().sum():,} ({df.duplicated().sum()/len(df)*100:.2f}%)')
print(f'  Komoditas Unik      : {df["komoditas"].nunique()}')
print(f'  Daerah Unik         : {df["daerah"].nunique()}')
print(f'  Sumber Unik         : {df["sumber"].nunique()}')
print(f'  Rentang Tanggal     : {df["tanggal"].min().date()} s/d {df["tanggal"].max().date()}')
print('='*60)

In [ ]:
# Tipe data per kolom
display(Markdown('### Tipe Data per Kolom'))
display(df.dtypes.to_frame('Tipe Data'))

In [ ]:
# Missing Values
display(Markdown('### Missing Values'))
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Persentase (%)': missing_pct})
display(missing_df)

---
## 4. Statistik Deskriptif

In [ ]:
# Statistik deskriptif untuk kolom numerik
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
records = []
for col in num_cols:
    s = df[col].dropna()
    if len(s) == 0:
        continue
    mode_val = s.mode()
    records.append({
        'Kolom': col, 'Count': int(s.count()),
        'Mean': round(s.mean(), 2), 'Median': round(s.median(), 2),
        'Modus': round(mode_val.iloc[0], 2) if len(mode_val) > 0 else np.nan,
        'Std Dev': round(s.std(), 2),
        'Min': round(s.min(), 2), 'Max': round(s.max(), 2),
        'Q25': round(s.quantile(0.25), 2),
        'Q75': round(s.quantile(0.75), 2),
        'Skewness': round(s.skew(), 4),
        'Kurtosis': round(s.kurtosis(), 4),
    })

stats_df = pd.DataFrame(records)
display(Markdown('### Statistik Deskriptif Kolom Numerik'))
display(stats_df)

---
## 5. Analisis Waktu (Temporal Analysis)

In [ ]:
tanggal = df['tanggal'].dropna()
date_range = pd.date_range(tanggal.min(), tanggal.max(), freq='D')
existing_dates = tanggal.dt.date.unique()
missing_dates = set(date_range.date) - set(existing_dates)
daily_count = df.groupby(df['tanggal'].dt.date).size()

print(f'Tanggal Awal          : {tanggal.min().date()}')
print(f'Tanggal Akhir         : {tanggal.max().date()}')
print(f'Total Hari Unik       : {len(existing_dates):,}')
print(f'Total Missing Dates   : {len(missing_dates):,}')
print(f'Rata-rata Update/Hari : {daily_count.mean():.1f} baris')

In [ ]:
# Jumlah data per tahun
yearly = df.groupby('tahun').size().reset_index(name='jumlah')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(yearly['tahun'].astype(int).astype(str), yearly['jumlah'],
              color='#4C72B0', edgecolor='white')
ax.set_title('Jumlah Data per Tahun', fontsize=16, fontweight='bold')
ax.set_xlabel('Tahun')
ax.set_ylabel('Jumlah Baris')
for bar, val in zip(bars, yearly['jumlah']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

---
## 6. Analisis Harga Komoditas

In [ ]:
# Statistik harga per komoditas
valid = df.dropna(subset=['harga_numeric'])

komod_stats = valid.groupby('komoditas')['harga_numeric'].agg(
    ['mean', 'median', 'min', 'max', 'std', 'count']
).round(0).reset_index()
komod_stats.columns = ['Komoditas', 'Rata-rata', 'Median', 'Min', 'Max', 'Std Dev', 'Jumlah Data']
komod_stats['Volatilitas (%)'] = ((komod_stats['Std Dev'] / komod_stats['Rata-rata']) * 100).round(2)
komod_stats = komod_stats.sort_values('Rata-rata', ascending=False)

display(Markdown('### Statistik Harga per Komoditas'))
display(komod_stats)

In [ ]:
# Bar chart rata-rata harga
fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette('viridis', len(komod_stats))
bars = ax.barh(komod_stats['Komoditas'], komod_stats['Rata-rata'], color=colors)
ax.set_title('Rata-rata Harga per Komoditas (2021-2026)', fontsize=16, fontweight='bold')
ax.set_xlabel('Harga (Rp)')
for bar, val in zip(bars, komod_stats['Rata-rata']):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'Rp {val:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot harga per komoditas
fig, ax = plt.subplots(figsize=(14, 7))
order = valid.groupby('komoditas')['harga_numeric'].median().sort_values(ascending=False).index
sns.boxplot(data=valid, x='komoditas', y='harga_numeric', order=order, ax=ax, palette='Set2')
ax.set_title('Distribusi Harga per Komoditas (Boxplot)', fontsize=16, fontweight='bold')
ax.set_xlabel('Komoditas')
ax.set_ylabel('Harga (Rp)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram distribusi harga
fig, ax = plt.subplots(figsize=(12, 6))
data_h = df['harga_numeric'].dropna()
ax.hist(data_h, bins=50, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.set_title('Distribusi Harga Seluruh Komoditas', fontsize=16, fontweight='bold')
ax.set_xlabel('Harga (Rp)')
ax.set_ylabel('Frekuensi')
ax.axvline(data_h.mean(), color='red', linestyle='--', label=f'Mean: Rp {data_h.mean():,.0f}')
ax.axvline(data_h.median(), color='green', linestyle='--', label=f'Median: Rp {data_h.median():,.0f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Perubahan harga harian per komoditas
change_records = []
for kom in valid['komoditas'].unique():
    sub = valid[valid['komoditas'] == kom]
    daily_avg = sub.groupby(sub['tanggal'].dt.date)['harga_numeric'].mean().sort_index()
    if len(daily_avg) < 2:
        continue
    pct = daily_avg.pct_change().dropna()
    change_records.append({
        'Komoditas': kom,
        'Avg Daily Change (%)': round(pct.mean() * 100, 4),
        'Max Increase (%)': round(pct.max() * 100, 2),
        'Max Decrease (%)': round(pct.min() * 100, 2),
    })
change_df = pd.DataFrame(change_records).sort_values('Avg Daily Change (%)', ascending=False)

display(Markdown('### Perubahan Harga Harian per Komoditas'))
display(change_df)

In [ ]:
# Volatilitas harga
sorted_vol = komod_stats.sort_values('Volatilitas (%)', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if v > 50 else '#3498db' for v in sorted_vol['Volatilitas (%)']]
ax.barh(sorted_vol['Komoditas'], sorted_vol['Volatilitas (%)'], color=colors)
ax.set_title('Volatilitas Harga per Komoditas (CV %)', fontsize=16, fontweight='bold')
ax.set_xlabel('Volatilitas (%)')
plt.tight_layout()
plt.show()

---
## 7. Tren Harga (Time Series)

In [ ]:
# Tren harga bulanan per komoditas
monthly = valid.groupby([valid['tanggal'].dt.to_period('M'), 'komoditas'])['harga_numeric'].mean().reset_index()
monthly['tanggal'] = monthly['tanggal'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 8))
for kom in monthly['komoditas'].unique():
    sub = monthly[monthly['komoditas'] == kom]
    ax.plot(sub['tanggal'], sub['harga_numeric'], label=kom, linewidth=1.8)

ax.set_title('Tren Harga Bulanan per Komoditas (2021-2026)', fontsize=16, fontweight='bold')
ax.set_xlabel('Waktu')
ax.set_ylabel('Harga Rata-rata (Rp)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 8. Analisis Wilayah

In [ ]:
# Statistik harga per wilayah
region_stats = valid.groupby('daerah')['harga_numeric'].agg(
    ['mean', 'median', 'min', 'max', 'count']
).round(0).reset_index()
region_stats.columns = ['Daerah', 'Rata-rata', 'Median', 'Min', 'Max', 'Jumlah Data']
region_stats = region_stats.sort_values('Rata-rata', ascending=False)

display(Markdown('### Statistik Harga per Wilayah'))
display(region_stats)

In [ ]:
# Heatmap: Komoditas vs Daerah
cross = valid.pivot_table(
    values='harga_numeric', index='komoditas', columns='daerah', aggfunc='mean'
).round(0)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cross, annot=True, fmt=',.0f', cmap='YlGnBu', linewidths=0.5, ax=ax)
ax.set_title('Rata-rata Harga: Komoditas vs Daerah', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Missing Value Analysis

In [ ]:
# Heatmap missing values (sample)
fig, ax = plt.subplots(figsize=(14, 5))
sample = df.isnull().astype(int).sample(min(500, len(df)), random_state=42)
sns.heatmap(sample.T, cbar=True, yticklabels=True, cmap='YlOrRd', ax=ax)
ax.set_title('Missing Value Heatmap (500 rows sample)', fontsize=16, fontweight='bold')
ax.set_xlabel('Row Index (sampled)')
plt.tight_layout()
plt.show()

---
## 10. Ringkasan & Kesimpulan

### Temuan Utama:

1. **Volume Data:** 329.460 baris data dari 6 file JSON (2021-2026)
2. **Kualitas Data:** Sangat baik, hanya 0.09% data duplikat dan tidak ada missing values pada kolom utama
3. **Komoditas Paling Mahal:** Daging Sapi dengan rata-rata harga tertinggi
4. **Komoditas Paling Volatil:** Cabai Merah dan Cabai Rawit menunjukkan fluktuasi harga tertinggi
5. **Cakupan Wilayah:** 3 kota/kabupaten (Banda Aceh, Lhokseumawe, Meulaboh)
6. **Sumber Data:** 4 sumber (Pasar Modern, Pasar Tradisional, Pedagang Besar, Produsen)

In [ ]:
print('='*60)
print('  EDA SELESAI')
print(f'  Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('='*60)